# 1. 필요한 Import

In [1]:
!pip install kneed

In [1]:
from tqdm import tqdm
import warnings, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch import linalg as LA
from itertools import permutations
from scipy.sparse import csr_matrix

from collections import Counter, defaultdict
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from kneed import KneeLocator
warnings.filterwarnings('ignore')

# 2. 데이터 수집 및 전처리

In [2]:
dir = './spotify_data/'

df_ratings = pd.read_csv(dir + 'ratings.csv', usecols=['user_id', 'track_id', 'rating'], encoding='cp949')
df_tracks = pd.read_csv(dir + 'items.csv', usecols=['track_id', 'artist_name', 'track_name', 'playlistname'], encoding='latin-1')
print(df_ratings.shape)
print(df_tracks.shape)

In [3]:
df_ratings = df_ratings.groupby('user_id').filter(lambda x:len(x)>=50)
df_ratings = df_ratings.groupby('track_id').filter(lambda x:len(x)>=50)
df_tracks = df_tracks[df_tracks.track_id.isin(df_ratings.track_id.unique())]
print(df_ratings.shape)
print(df_tracks.shape)

(3451185, 3)
(27818, 4)


# 3. Input Matrix 제작

In [4]:
num_tracks = df_tracks.track_id.nunique()
num_users = df_ratings.user_id.nunique()

track_map = dict(zip(df_ratings.track_id.unique(), range(num_tracks)))
df_ratings['track_id'] = df_ratings['track_id'].map(track_map)
df_tracks['track_id']  = df_tracks['track_id'].map(track_map)

user_map = dict(zip(df_ratings.user_id.unique(), range(num_users)))
df_ratings['user_id'] = df_ratings['user_id'].map(user_map)

Matrix_T = csr_matrix(
    (df_ratings['rating'].values / 5,
     (df_ratings['track_id'].values,
      df_ratings['user_id'].values)),
    shape=(num_tracks, num_users)
)

def create_train_test_masks(Matrix_T, test_ratio=0.2, seed=42):
    R, C = Matrix_T.nonzero()
    ratings = Matrix_T.data
    
    num_positives = len(R)
    
    np.random.seed(seed)
    indices = np.arange(num_positives)
    
    train_indices, test_indices = train_test_split(
        indices,
        test_size=test_ratio,
        random_state=seed,
        shuffle=True
    )
    
    R_train = R[train_indices]
    C_train = C[train_indices]
    Ratings_train = ratings[train_indices]
    
    Matrix_T_train = csr_matrix(
        (Ratings_train, (R_train, C_train)),
        shape=Matrix_T.shape
    )
    
    R_test = R[test_indices]
    C_test = C[test_indices]
    Ratings_test = ratings[test_indices]
    
    Matrix_T_test = csr_matrix(
        (Ratings_test, (R_test, C_test)),
        shape=Matrix_T.shape
    )

    return Matrix_T_train, Matrix_T_test

Matrix_T_train, Matrix_T_valid = create_train_test_masks(Matrix_T, test_ratio=0.2, seed=42)

class DAETrainDataset(Dataset):
    def __init__(self, train_mask: csr_matrix):
        self.data = torch.from_numpy(train_mask.toarray()).float()
        
    def __len__(self):
        return self.data.size(0)
        
    def __getitem__(self, idx):
        X = self.data[idx]
        Y = self.data[idx]
        return X, Y

train_dataset = DAETrainDataset(train_mask=Matrix_T_train)
train_loader = DataLoader(
    train_dataset, 
    batch_size=256, 
    shuffle=True
)

class DAERankingDataset(Dataset):
    def __init__(self, train_mask: csr_matrix, test_mask: csr_matrix):
        self.x_data = torch.from_numpy(train_mask.toarray()).float()
        self.y_data = torch.from_numpy(test_mask.toarray()).float()
        
    def __len__(self):
        return self.x_data.size(0)
        
    def __getitem__(self, idx):
        return self.x_data[idx], self.y_data[idx]

valid_dataset = DAERankingDataset(
    train_mask=Matrix_T_train,
    test_mask=Matrix_T_valid
)

validation_loader = DataLoader(
    valid_dataset, 
    batch_size=64,
    shuffle=False
)

# 4. Model 제작

In [5]:
class Net(nn.Module):
    def __init__(self, D, K, dropout_rate=0.5):
        super(Net, self).__init__()
        self.encoder = nn.Linear(D, K, bias=False)
        self.decoder = nn.Linear(K, D)
        self.dropout = nn.Dropout(p=dropout_rate)
        
        self._init_weights()
        
    def _init_weights(self):
        nn.init.xavier_uniform_(self.encoder.weight)
        nn.init.xavier_uniform_(self.decoder.weight)
    
    def forward(self, x):
        x = torch.relu(self.encoder(x))
        x = self.dropout(x)
        x = torch.sigmoid(self.decoder(x))
        return x
        
def train(model, criterion, train_loader, validation_loader, optimizer, epochs=50, lambda_orthogonality = 0.001):
    validation_losses = []  
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        
        for x, y in train_loader:
            optimizer.zero_grad()
            z = model(x)
            loss = criterion(z, y)
            
            W_Encoder = model.encoder.weight
            H = W_Encoder.size(0)
            
            gram_matrix = torch.matmul(W_Encoder, W_Encoder.T)
            I = torch.eye(H)
            loss_orthogonality = LA.matrix_norm(gram_matrix - I)**2
            
            loss = loss + lambda_orthogonality * loss_orthogonality
            
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
        
        model.eval()
        valid_loss = 0.0
        for x, y in validation_loader:
            z = model(x)
            valid_loss += criterion(z, y).item()
        
        valid_loss = valid_loss / len(validation_loader)
        validation_losses.append(valid_loss)
        
        print(f"Epoch {epoch+1}/{epochs}: Train Loss: {running_loss/len(train_loader):.4f}, "
              f"Validation Loss: {valid_loss:.4f}, Orthogonality Loss: {loss_orthogonality.item():.6f}")
    
    return validation_losses

criterion = nn.MSELoss() 
hidden_dim = 32
epochs = 50
learning_rate = 0.01
lambda_orthogonality = 0.01
dropout_rate = 0.5
model = Net(num_users, hidden_dim, dropout_rate)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

print(f"학습 시작! Latent Vector 차원 K={hidden_dim}")
training_results = train(model, criterion, train_loader, validation_loader, optimizer, epochs=epochs, lambda_orthogonality=0.01)
print(f"학습 완료!")

학습 시작! Latent Vector 차원 K=32
Epoch 1/50: Train Loss: 0.0643, Validation Loss: 0.0114, Orthogonality Loss: 0.166491
Epoch 2/50: Train Loss: 0.0125, Validation Loss: 0.0055, Orthogonality Loss: 0.051974
Epoch 3/50: Train Loss: 0.0078, Validation Loss: 0.0033, Orthogonality Loss: 0.028059
Epoch 4/50: Train Loss: 0.0061, Validation Loss: 0.0024, Orthogonality Loss: 0.013304
Epoch 5/50: Train Loss: 0.0051, Validation Loss: 0.0020, Orthogonality Loss: 0.010024
Epoch 6/50: Train Loss: 0.0046, Validation Loss: 0.0017, Orthogonality Loss: 0.007271
Epoch 7/50: Train Loss: 0.0043, Validation Loss: 0.0015, Orthogonality Loss: 0.002569
Epoch 8/50: Train Loss: 0.0040, Validation Loss: 0.0013, Orthogonality Loss: 0.005913
Epoch 9/50: Train Loss: 0.0039, Validation Loss: 0.0012, Orthogonality Loss: 0.002279
Epoch 10/50: Train Loss: 0.0038, Validation Loss: 0.0012, Orthogonality Loss: 0.004355
Epoch 11/50: Train Loss: 0.0037, Validation Loss: 0.0011, Orthogonality Loss: 0.003187
Epoch 12/50: Train Loss

# 5. Encoder Matrix 수직성 검증

In [6]:
W_Enc_matrix = model.encoder.weight.data.cpu().numpy()
W_Encoder_tensor = torch.from_numpy(W_Enc_matrix).float()
norm_check = LA.norm(W_Encoder_tensor, ord=2, dim=1)
norm_sorted = np.sort(norm_check.numpy())[::-1]

print("순위 | Norm")
print("-" * 20)
for i, norm in enumerate(norm_sorted):
    if i < 10:
        print(f" {i+1:2d} | {norm:.4f} (상위)")
    elif i >= len(norm_sorted) - 5:
        if i == len(norm_sorted) - 5:
            print("...")
        if i >= len(norm_sorted) - 5 and i > 10:
            print(f" {i+1:2d} | {norm:.4f} (하위)")
  
gram_test = torch.matmul(W_Encoder_tensor, W_Encoder_tensor.T)
K = W_Encoder_tensor.size(0)
I_test = torch.eye(K)

error_orthogonality = (LA.matrix_norm(gram_test - I_test)**2).item()
print(f"\n최종 직교 오차: {error_orthogonality:.6f}")
max_error = torch.max(gram_test - I_test)
print(f"W x W^T와 I 중 가장 차이 큰 값: {max_error:.6f}")

print("\nW x W^T 6 x 6까지:")
print(np.around(gram_test[:6, :6].numpy(), decimals=4))

순위 | Norm
--------------------
  1 | 1.0000 (상위)
  2 | 1.0000 (상위)
  3 | 1.0000 (상위)
  4 | 1.0000 (상위)
  5 | 1.0000 (상위)
  6 | 1.0000 (상위)
  7 | 1.0000 (상위)
  8 | 1.0000 (상위)
  9 | 1.0000 (상위)
 10 | 1.0000 (상위)
...
 28 | 1.0000 (하위)
 29 | 1.0000 (하위)
 30 | 1.0000 (하위)
 31 | 1.0000 (하위)
 32 | 1.0000 (하위)

최종 직교 오차: 0.000000
W x W^T와 I 중 가장 차이 큰 값: 0.000062

W x W^T 6 x 6까지:
[[ 1.      0.      0.     -0.      0.      0.    ]
 [ 0.      1.0001  0.      0.      0.      0.    ]
 [ 0.      0.      1.0001  0.      0.      0.    ]
 [-0.      0.      0.      1.      0.      0.    ]
 [ 0.      0.      0.      0.      1.      0.    ]
 [ 0.      0.      0.      0.      0.      1.    ]]


# 6. 음원의 유사도 결과

In [7]:
dense_matrix = torch.tensor(Matrix_T.toarray(), dtype=torch.float32)
track_representations = model.encoder(dense_matrix)

latent_tensors = torch.relu(track_representations)

N = random.randint(0, num_tracks-1)
trackN_embedding = latent_tensors[N]

trackN_embedding_vector = trackN_embedding.view(1, -1)

def pearson_correlation(x, Y):
    mean_x = torch.mean(x, dim=1, keepdim = True)
    mean_Y = torch.mean(Y, dim=1, keepdim = True)
    
    centered_x = x - mean_x
    centered_Y = Y - mean_Y
    
    covariances = torch.mm(centered_Y, centered_x.T)
    std_products = LA.norm(centered_x, dim=1, keepdim = True) * LA.norm(centered_Y, dim=1, keepdim=True)
    
    return (covariances / std_products + 1e-8).squeeze(1)


similarity = pearson_correlation(trackN_embedding_vector, latent_tensors)

top_N = 11
top_scores, top_indices = torch.topk(similarity, k = top_N)

top_scores = top_scores.detach().numpy()
top_indices = top_indices.numpy()

recommendations = df_tracks[df_tracks['track_id'].isin(top_indices[1:])]

top_scores_pd = pd.Series(top_scores, index=top_indices, name='similarity')
recommendations['similarity'] = recommendations['track_id'].map(top_scores_pd)

recommendations = recommendations.sort_values(by='similarity', ascending=False)

print("track_id ", N,"의 유사도 기반 추천")
print(recommendations[['track_id', 'artist_name', 'track_name', 'playlistname', 'similarity']])

track_id  21640 의 유사도 기반 추천
       track_id artist_name       track_name         playlistname  similarity
46549     18298     Sublime  Under My Voodoo  Sublime â Sublime    0.991270
46539     18293     Sublime     Garden Grove  Sublime â Sublime    0.988517
46536     21638     Sublime         Burritos  Sublime â Sublime    0.988112
46541     18294     Sublime        Jailhouse  Sublime â Sublime    0.984308
46546     18296     Sublime  Same In The End  Sublime â Sublime    0.982327
46543     21639     Sublime       Paddle Out  Sublime â Sublime    0.973345
46547     18297     Sublime             Seed  Sublime â Sublime    0.968733
46544     18295     Sublime        Pawn Shop  Sublime â Sublime    0.933717
46540     12496     Sublime        Get Ready  Sublime â Sublime    0.911004
85272     18355     Sublime       Don't Push              Starred    0.894004


# 7. K-Fold Cross Validation

In [13]:
def compute_rmse(model, dataloader):
    model.eval()
    mse_sum = 0
    count = 0
    criterion = torch.nn.MSELoss(reduction='sum')

    with torch.no_grad():
        for x, y in dataloader:
            recon = model(x)
            mse_sum += criterion(recon, x).item()
            count += x.numel()

    rmse = np.sqrt(mse_sum / count)
    return rmse

print("STARTING K-FOLD CROSS VALIDATION K=3")

kf = KFold(n_splits=3, shuffle=True, random_state=42)
fold_results_rmse = []
full_matrix = Matrix_T 

for fold, (train_idx, val_idx) in enumerate(kf.split(full_matrix)):
    print(f" FOLD {fold+1}")


    matrix_fold_train = full_matrix[train_idx]
    matrix_fold_valid = full_matrix[val_idx]
    
    train_dataset = DAETrainDataset(train_mask=Matrix_T_train)
    train_loader = DataLoader(
        train_dataset, 
        batch_size=256, 
        shuffle=True
    )
    valid_dataset = DAERankingDataset(
        train_mask=Matrix_T_train,
        test_mask=Matrix_T_valid
    )

    validation_loader = DataLoader(
        valid_dataset, 
        batch_size=64,
        shuffle=False
    )

    model_fold = Net(num_users, 32, dropout_rate=0.5)

    optimizer_fold = torch.optim.Adam(model_fold.parameters(), lr=0.05)
    criterion_fold = nn.MSELoss()


    train(model_fold, criterion_fold, train_loader, validation_loader, optimizer_fold, epochs=5, lambda_orthogonality=0.01)

    rmse_val = compute_rmse(model_fold, validation_loader)
    print(f"Fold {fold+1} Final RMSE: {rmse_val:.4f}")
    fold_results_rmse.append(rmse_val)   

print("\n" + "="*30)
print("K-FOLD RESULTS SUMMARY")
print("="*30)
print(f"RMSE per fold: {fold_results_rmse}")
print(f"Average RMSE: {np.mean(fold_results_rmse):.4f}")

STARTING K-FOLD CROSS VALIDATION K=3
 FOLD 1
Epoch 1/5: Train Loss: 4.1407, Validation Loss: 0.0013, Orthogonality Loss: 0.006988
Epoch 2/5: Train Loss: 0.0037, Validation Loss: 0.0011, Orthogonality Loss: 0.000703
Epoch 3/5: Train Loss: 0.0036, Validation Loss: 0.0010, Orthogonality Loss: 0.000471
Epoch 4/5: Train Loss: 0.0035, Validation Loss: 0.0010, Orthogonality Loss: 0.000135
Epoch 5/5: Train Loss: 0.0034, Validation Loss: 0.0009, Orthogonality Loss: 0.000083
Fold 1 Final RMSE: 0.0577
 FOLD 2
Epoch 1/5: Train Loss: 4.1372, Validation Loss: 0.0014, Orthogonality Loss: 0.007038
Epoch 2/5: Train Loss: 0.0037, Validation Loss: 0.0011, Orthogonality Loss: 0.000633
Epoch 3/5: Train Loss: 0.0036, Validation Loss: 0.0010, Orthogonality Loss: 0.000538
Epoch 4/5: Train Loss: 0.0035, Validation Loss: 0.0010, Orthogonality Loss: 0.000164
Epoch 5/5: Train Loss: 0.0034, Validation Loss: 0.0009, Orthogonality Loss: 0.000133
Fold 2 Final RMSE: 0.0577
 FOLD 3
Epoch 1/5: Train Loss: 4.1499, Valida

# 8. 새로운 클러스터링 실험

In [9]:
dominant_factors = torch.argmax(track_representations, dim=1)

ortho_clusters = defaultdict(list)
for track_id, factor_id in enumerate(dominant_factors):
    ortho_clusters[factor_id.item()].append(track_id)
ortho_clusters = dict(ortho_clusters)

print(f"\n수직 요소 기반 클러스터링: 총 {len(ortho_clusters)} 클러스터")
for cid in sorted(ortho_clusters.keys())[:5]:
    print(f"  클러스터 {cid + 1}번: {len(ortho_clusters[cid])} 트랙")

min_cluster_id = min(ortho_clusters, key=lambda k: len(ortho_clusters[k]))
min_cluster_size = len(ortho_clusters[min_cluster_id])
print(f"\n  가장 원소의 개수가 작은 클러스터는 {min_cluster_id+1}번 : {min_cluster_size} tracks")


수직 요소 기반 클러스터링: 총 32 클러스터
  클러스터 1번: 1068 트랙
  클러스터 2번: 1428 트랙
  클러스터 3번: 267 트랙
  클러스터 4번: 2867 트랙
  클러스터 5번: 671 트랙

  가장 원소의 개수가 작은 클러스터는 15번 : 31 tracks


# 9. 클러스터링 Helper Function

In [7]:
def merge_clusters_by_similarity(clusters, W_Enc_matrix, target_k):
    
    W_normalized = F.normalize(W_Encoder_tensor, p=2, dim=1)

    similarity_matrix = torch.matmul(W_normalized, W_normalized.T).numpy()
    np.fill_diagonal(similarity_matrix, 0)
    
    current_clusters = {cid: list(tracks) for cid, tracks in clusters.items()}
    active_clusters = set(current_clusters.keys())
    
    while len(active_clusters) > target_k:
        max_similarity = -1
        merge_pair = None
        
        active_list = list(active_clusters)
        for i in range(len(active_list)):
            for j in range(i + 1, len(active_list)):
                cid_i, cid_j = active_list[i], active_list[j]
                sim = similarity_matrix[cid_i, cid_j]
                if sim > max_similarity:
                    max_similarity = sim
                    merge_pair = (cid_i, cid_j)
        
        if merge_pair is None:
            break
        
        cid_i, cid_j = merge_pair
        current_clusters[cid_i].extend(current_clusters[cid_j])
        del current_clusters[cid_j]
        active_clusters.remove(cid_j)
    
    merged_clusters = {}
    for new_id, old_id in enumerate(sorted(active_clusters)):
        merged_clusters[new_id] = current_clusters[old_id]
    
    return merged_clusters

def create_cluster_labels(clusters, n_samples):

    labels = np.zeros(n_samples, dtype=int)
    for cluster_id, track_list in clusters.items():
        for track_id in track_list:
            if track_id < n_samples:
                labels[track_id] = cluster_id
    return labels

def compute_cluster_balance(clusters):

    sizes = [len(tracks) for tracks in clusters.values()]
    
    cv = np.std(sizes) / np.mean(sizes)
    
    sizes_sorted = np.sort(sizes)
    n = len(sizes_sorted)
    gini = (2 * np.sum((np.arange(n) + 1) * sizes_sorted)) / (n * np.sum(sizes_sorted)) - (n + 1) / n
    
    return {
        'min': min(sizes),
        'max': max(sizes),
        'mean': np.mean(sizes),
        'std': np.std(sizes),
        'cv': cv,
        'gini': gini
    }

def compute_artist_diversity(clusters, df_tracks):

    diversities = []
    for cluster_id, track_list in clusters.items():
        cluster_tracks = df_tracks[df_tracks['track_id'].isin(track_list)]
        if len(cluster_tracks) == 0:
            continue
        unique_artists = cluster_tracks['artist_name'].nunique()
        total_tracks = len(cluster_tracks)
        diversity = unique_artists / total_tracks
        diversities.append(diversity)
    
    return np.mean(diversities) if diversities else 0.0

In [6]:
def find_optimal_k_by_elbow(data, max_k):

    inertia_values = []
    k_range = range(1, max_k + 1)
    
    for k in k_range:
        kmeans = KMeans(
            n_clusters=k, 
            random_state=42, 
            n_init=10, 
            max_iter=300
        )
        kmeans.fit(data)
        inertia_values.append(kmeans.inertia_)
    
    k_array = np.array(k_range)

    kn = KneeLocator(
        k_array, 
        inertia_values, 
        direction='decreasing', 
        curve='convex',
        interp_method='polynomial'
    )
    
    optimal_k = kn.knee
    
    if optimal_k is not None:
        print("최적의 k는 ", optimal_k)
        return optimal_k
    else:
        print("최적의 k는 ", 32)
        return 32

# 10. 3가지 클러스터링 결과

In [12]:
def three_way_clustering_comparison(ortho_clusters, track_representations, 
                                   W_Enc_matrix, df_tracks):
    n_samples = len(track_representations)
    track_reps_np = track_representations.detach().numpy()
    
    print("1번 방식: 잠재 벡터 클러스터링")
    
    optimal_k = find_optimal_k_by_elbow(track_reps_np, max_k=32)
    
    kmeans_latent = KMeans(
        n_clusters=optimal_k, 
        random_state=42,
        n_init=10,
        max_iter=300
    )
    
    kmeans_latent_labels = kmeans_latent.fit_predict(track_reps_np)
    
    kmeans_latent_clusters = defaultdict(list)
    for track_id, cluster_id in enumerate(kmeans_latent_labels):
        kmeans_latent_clusters[cluster_id].append(track_id)
    kmeans_latent_clusters = dict(kmeans_latent_clusters)
    
    print(f"✓ 완료! 총 {len(kmeans_latent_clusters)} 클러스터")
    print(f"   분산도: {kmeans_latent.inertia_:.2f}")
    
    """-------------------------------------------------------"""
    
    
    print(f"\n2번 방식: 가장 큰 값 기준 클러스터 차원 줄이기")
    ortho_max_labels = np.zeros(n_samples, dtype=int)

    for factor_id, track_list in ortho_clusters.items():
        for track_id in track_list:
            if track_id < n_samples:
                ortho_max_labels[track_id] = factor_id
    
    print("✓ 완료! 총 32 클러스터")
    
    """-------------------------------------------------------"""
    
    print(f"\n3번 방식: 가장 큰 값 기준 클러스터 차원 줄이기 (32 → {optimal_k})...")
    
    ortho_merged = merge_clusters_by_similarity(
        ortho_clusters, W_Enc_matrix, target_k=optimal_k
    )
    ortho_labels = create_cluster_labels(ortho_merged, n_samples)
    
    print(f"✓ 완료! 총 {len(ortho_merged)} 클러스터")
    
    """-------------------------------------------------------"""

    print("\n" + "="*70)
    print("클러스터링 평가 지표")
    print("="*70)
    
    results = {}
    
    print("\n--- 실루엣 계수 (높은게 좋음) ---")
    
    kmeans_latent_sil = silhouette_score(track_reps_np, kmeans_latent_labels)
    ortho_max_sil = silhouette_score(track_reps_np, ortho_max_labels)
    ortho_sil = silhouette_score(track_reps_np, ortho_labels)
    
    
    print(f"Method 1 (K-means Latent):     {kmeans_latent_sil:.4f}")
    print(f"Method 2 (Orthogonal Max):     {ortho_max_sil:.4f}")
    print(f"Method 3 (Orthogonal): {ortho_sil:.4f}")
    
    if (ortho_sil > kmeans_latent_sil) and (ortho_sil > ortho_max_sil):
        winner_sil = 'Orthogonal'
    elif ortho_max_sil > kmeans_latent_sil:
        winner_sil = 'Orthogonal Max'
    else:
        winner_sil = 'K-means Latent'
    print(f"승자는 ✓ {winner_sil}")
    
    results['silhouette'] = {
        'kmeans_latent': kmeans_latent_sil,
        'orthogonal_max': ortho_max_sil,
        'orthogonal': ortho_sil,
        'winner': winner_sil
    }
    
    """-------------------------------------------------------"""
    
    print("\n--- Davies-Bouldin Index (작을수록 좋음) ---")
    
    kmeans_latent_db = davies_bouldin_score(track_reps_np, kmeans_latent_labels)
    ortho_max_db = davies_bouldin_score(track_reps_np, ortho_max_labels)
    ortho_db = davies_bouldin_score(track_reps_np, ortho_labels)
    
    
    print(f"Method 1 (K-means Latent):     {kmeans_latent_db:.4f}")
    print(f"Method 2 (Orthogonal Max):     {ortho_max_db:.4f}")
    print(f"Method 3 (Orthogonal): {ortho_db:.4f}")
    
    if (ortho_db < kmeans_latent_db) and (ortho_db < ortho_max_db):
        winner_db = 'Orthogonal'
    elif ortho_max_db < kmeans_latent_db:
        winner_db = 'Orthogonal Max'
    else:
        winner_db = 'K-means Latent'
    print(f"승자는 ✓ {winner_db}")
    
    results['davies_bouldin'] = {
        'kmeans_latent': kmeans_latent_db,
        'orthogonal_max': ortho_max_db,
        'orthogonal': ortho_db,
        'winner': winner_db
    }
    
    """-------------------------------------------------------"""
    
    print("\n--- Calinski-Harabasz Index (높을수록 좋음) ---")
    
    kmeans_latent_ch = calinski_harabasz_score(track_reps_np, kmeans_latent_labels)
    ortho_max_ch = calinski_harabasz_score(track_reps_np, ortho_max_labels)
    ortho_ch = calinski_harabasz_score(track_reps_np, ortho_labels)
    
    
    print(f"Method 1 (K-means Latent):     {kmeans_latent_ch:.4f}")
    print(f"Method 2 (Orthogonal Max):     {ortho_max_ch:.4f}")
    print(f"Method 3 (Orthogonal): {ortho_ch:.4f}")
    
    if (ortho_ch > kmeans_latent_ch) and (ortho_ch > ortho_max_ch):
        winner_ch = 'Orthogonal'
    elif ortho_max_ch > kmeans_latent_ch:
        winner_ch = 'Orthogonal Max'
    else:
        winner_ch = 'K-means Latent'
    print(f"승자는 ✓ {winner_ch}")
    
    results['calinski_harabasz'] = {
        'kmeans_latent': kmeans_latent_ch,
        'orthogonal_max': ortho_max_ch,
        'orthogonal': ortho_ch,
        'winner': winner_ch
    }
    
    """-------------------------------------------------------"""
    
    print("\n--- 클러스터 상세 지표 ---")
    
    kmeans_latent_balance = compute_cluster_balance(kmeans_latent_clusters)
    ortho_max_balance = compute_cluster_balance(ortho_clusters)
    ortho_balance = compute_cluster_balance(ortho_merged)
    
    print(f"\nMethod 1 (K-means Latent):")
    print(f"  Size range: {kmeans_latent_balance['min']} - {kmeans_latent_balance['max']}")
    print(f"  Mean ± Std: {kmeans_latent_balance['mean']:.1f} ± {kmeans_latent_balance['std']:.1f}")
    print(f"  CV: {kmeans_latent_balance['cv']:.4f}")
    print(f"  Gini: {kmeans_latent_balance['gini']:.4f}")
    
    print(f"\nMethod 2 (Orthogonal Max):")
    print(f"  Size range: {ortho_max_balance['min']} - {ortho_max_balance['max']}")
    print(f"  Mean ± Std: {ortho_max_balance['mean']:.1f} ± {ortho_max_balance['std']:.1f}")
    print(f"  CV: {ortho_max_balance['cv']:.4f}")
    print(f"  Gini: {ortho_max_balance['gini']:.4f}")
    
    print(f"\nMethod 3 (Orthogonal):")
    print(f"  Size range: {ortho_balance['min']} - {ortho_balance['max']}")
    print(f"  Mean ± Std: {ortho_balance['mean']:.1f} ± {ortho_balance['std']:.1f}")
    print(f"  CV: {ortho_balance['cv']:.4f}")
    print(f"  Gini: {ortho_balance['gini']:.4f}")
    
    print("\n--- 변동 계수 (작을수록 좋음) ---")
    if (ortho_balance['cv'] < kmeans_latent_balance['cv']) and (ortho_balance['cv'] < ortho_max_balance['cv']):
        winner_cv = 'Orthogonal'
    elif ortho_max_balance['cv'] < kmeans_latent_balance['cv']:
        winner_cv = 'Orthogonal Max'
    else:
        winner_cv = 'K-means Latent'
    print(f"승자는 ✓ {winner_cv}")
    
    print("\n--- 지니 계수 (작을수록 좋음) ---")
    if (ortho_balance['gini'] < kmeans_latent_balance['gini']) and (ortho_balance['gini'] < ortho_max_balance['gini']):
        winner_gini = 'Orthogonal'
    elif ortho_max_balance['gini'] < kmeans_latent_balance['gini']:
        winner_gini = 'Orthogonal Max'
    else:
        winner_gini = 'K-means Latent'
    print(f"승자는 ✓ {winner_gini}")
    
    results['balance'] = {
        'kmeans_latent': kmeans_latent_balance,
        'orthogonal_max': ortho_max_balance,
        'orthogonal': ortho_balance,
        'winner_cv': winner_cv,
        'winner_gini': winner_gini
    }
    
    """-------------------------------------------------------"""
    
    print("\n--- 아티스트 다양성 (higher is better) ---")
    
    kmeans_latent_diversity = compute_artist_diversity(kmeans_latent_clusters, df_tracks)
    ortho_max_diversity = compute_artist_diversity(ortho_clusters, df_tracks)
    ortho_diversity = compute_artist_diversity(ortho_merged, df_tracks)
    
    
    print(f"Method 1 (K-means Latent):     {kmeans_latent_diversity:.4f}")
    print(f"Method 2 (Orthogonal Max):     {ortho_max_diversity:.4f}")
    print(f"Method 3 (Orthogonal): {ortho_diversity:.4f}")
    
    if (ortho_diversity > kmeans_latent_diversity) and (ortho_diversity > ortho_max_diversity):
        winner_diversity = 'Orthogonal'
    elif ortho_max_diversity > kmeans_latent_diversity:
        winner_diversity = 'Orthogonal Max'
    else:
        winner_diversity = 'K-means Latent'
    print(f"승자는 ✓ {winner_diversity}")
    
    results['artist_diversity'] = {
        'kmeans_latent': kmeans_latent_diversity,
        'orthogonal_max': ortho_max_diversity,
        'orthogonal': ortho_diversity,
        'winner': winner_diversity
    }
    
    return {
        'kmeans_latent_clusters': kmeans_latent_clusters,
        'orthogonal_max_clusters': ortho_clusters,
        'orthogonal_clusters': ortho_merged,
        'results': results
    }

In [13]:
comparison_results = three_way_clustering_comparison(
    ortho_clusters=ortho_clusters,
    track_representations=track_representations,
    W_Enc_matrix=W_Enc_matrix,
    df_tracks=df_tracks
)

print("\n✓ 평가 지표 완료")

1번 방식: 잠재 벡터 클러스터링
최적의 k는  7
✓ 완료! 총 7 클러스터
   분산도: 48351.73

2번 방식: 가장 큰 값 기준 클러스터 차원 줄이기
✓ 완료! 총 32 클러스터

3번 방식: 가장 큰 값 기준 클러스터 차원 줄이기 (32 → 7)...
✓ 완료! 총 7 클러스터

클러스터링 평가 지표

--- 실루엣 계수 (높은게 좋음) ---
Method 1 (K-means Latent):     0.1473
Method 2 (Orthogonal Max):     -0.1347
Method 3 (Orthogonal): -0.1679
승자는 ✓ K-means Latent

--- Davies-Bouldin Index (작을수록 좋음) ---
Method 1 (K-means Latent):     2.0228
Method 2 (Orthogonal Max):     4.7430
Method 3 (Orthogonal): 4.8283
승자는 ✓ K-means Latent

--- Calinski-Harabasz Index (높을수록 좋음) ---
Method 1 (K-means Latent):     2395.8128
Method 2 (Orthogonal Max):     111.8093
Method 3 (Orthogonal): 149.4326
승자는 ✓ K-means Latent

--- 클러스터 상세 지표 ---

Method 1 (K-means Latent):
  Size range: 433 - 14566
  Mean ± Std: 3974.0 ± 4535.6
  CV: 1.1413
  Gini: 0.5338

Method 2 (Orthogonal Max):
  Size range: 31 - 3467
  Mean ± Std: 869.3 ± 717.9
  CV: 0.8259
  Gini: 0.3889

Method 3 (Orthogonal):
  Size range: 31 - 16467
  Mean ± Std: 3974.0 ± 5871.1
  CV: 

# 11. 음원 특징 클러스터링 확인용

In [8]:
df_tracks2_clustering = pd.read_csv(dir + 'extra_items.csv', usecols=['track_id', 'danceability', 'energy', 'liveness', 'acousticness', 'valence'], encoding='latin-1')

cols = ['danceability', 'energy', 'liveness', 'acousticness', 'valence']

for col in cols:
    df_tracks2_clustering[col] = pd.to_numeric(df_tracks2_clustering[col], errors='coerce')
    df_tracks2_clustering[col] = df_tracks2_clustering[col].fillna(0.5)
    df_tracks2_clustering[col] = df_tracks2_clustering[col].clip(lower=0.0, upper=1.0)

n_samples = len(df_tracks2_clustering)
track_reps_np = df_tracks2_clustering[cols].values

optimal_k = find_optimal_k_by_elbow(track_reps_np, max_k=32)
    
kmeans_latent = KMeans(
    n_clusters=optimal_k, 
    random_state=42,
    n_init=10,
    max_iter=300
    )
    
kmeans_latent_labels = kmeans_latent.fit_predict(track_reps_np)
    
kmeans_latent_clusters = defaultdict(list)
for track_id, cluster_id in enumerate(kmeans_latent_labels):
    kmeans_latent_clusters[cluster_id].append(track_id)
kmeans_latent_clusters = dict(kmeans_latent_clusters)
    
print(f"✓ 완료! 총 {len(kmeans_latent_clusters)} 클러스터")
print(f"   분산도: {kmeans_latent.inertia_:.2f}")

print("\n--- 실루엣 계수 ---")
kmeans_latent_sil = silhouette_score(track_reps_np, kmeans_latent_labels)
print(f"다른 데이터셋 K-Means:     {kmeans_latent_sil:.4f}")

print("\n--- Davies-Bouldin Index ---")
kmeans_latent_db = davies_bouldin_score(track_reps_np, kmeans_latent_labels)
print(f"다른 데이터셋 K-Means:     {kmeans_latent_db:.4f}")

print("\n--- Calinski-Harabasz Index ---")    
kmeans_latent_ch = calinski_harabasz_score(track_reps_np, kmeans_latent_labels)
print(f"다른 데이터셋 K-Means:     {kmeans_latent_ch:.4f}")
    

print("\n--- 클러스터 상세 지표 ---")
kmeans_latent_balance = compute_cluster_balance(kmeans_latent_clusters)
print(f"  다른 데이터셋 K-Means:")
print(f"  Size range: {kmeans_latent_balance['min']} - {kmeans_latent_balance['max']}")
print(f"  Mean ± Std: {kmeans_latent_balance['mean']:.1f} ± {kmeans_latent_balance['std']:.1f}")
print(f"  CV: {kmeans_latent_balance['cv']:.4f}")
print(f"  Gini: {kmeans_latent_balance['gini']:.4f}")

print("\n--- 아티스트 다양성 ---")
kmeans_latent_diversity = compute_artist_diversity(kmeans_latent_clusters, df_tracks)
print(f"다른 데이터셋 K-Means:     {kmeans_latent_diversity:.4f}")

print("\n✓ 평가 지표 완료")

최적의 k는  5
✓ 완료! 총 5 클러스터
   분산도: 11135.37

--- 실루엣 계수 ---
다른 데이터셋 K-Means:     0.2728

--- Davies-Bouldin Index ---
다른 데이터셋 K-Means:     1.1983

--- Calinski-Harabasz Index ---
다른 데이터셋 K-Means:     37819.7473

--- 클러스터 상세 지표 ---
  다른 데이터셋 K-Means:
  Size range: 6539 - 27149
  Mean ± Std: 17902.6 ± 7544.1
  CV: 0.4214
  Gini: 0.2304

--- 아티스트 다양성 ---
다른 데이터셋 K-Means:     0.0000

✓ 평가 지표 완료
